# 📱 Deteksi Kecanduan Smartphone
## Notebook 3: Pemodelan & Training – XGBoost
---
Input  : `X_train.csv`, `y_train.csv`, `X_test.csv`, `y_test.csv`  
Output : `xgboost_model.pkl` + laporan tuning hyperparameter

### 3.1 Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score

print(f'✅ XGBoost version: {xgb.__version__}')

### 3.2 Load Data

In [ ]:
X_train = pd.read_csv('X_train.csv')
X_test  = pd.read_csv('X_test.csv')
y_train = pd.read_csv('y_train.csv').squeeze()
y_test  = pd.read_csv('y_test.csv').squeeze()

# XGBoost butuh label mulai dari 0
y_train_xgb = y_train - 1
y_test_xgb  = y_test  - 1

print(f'X_train shape : {X_train.shape}')
print(f'X_test  shape : {X_test.shape}')
print(f'Kelas target  : {sorted(y_train.unique())} → digeser ke {sorted(y_train_xgb.unique())} (0-indexed)')

### 3.3 Baseline Model XGBoost

In [ ]:
xgb_base = XGBClassifier(
    objective='multi:softmax',
    num_class=5,
    random_state=42,
    eval_metric='mlogloss',
    use_label_encoder=False,
    verbosity=0
)

xgb_base.fit(X_train, y_train_xgb)
y_pred_base = xgb_base.predict(X_test)
acc_base = accuracy_score(y_test_xgb, y_pred_base)

print(f'✅ Baseline XGBoost Accuracy: {acc_base:.4f} ({acc_base*100:.2f}%)')

### 3.4 Cross-Validation Baseline (5-Fold Stratified)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(xgb_base, X_train, y_train_xgb, cv=cv, scoring='accuracy', n_jobs=-1)

print('=== 5-Fold Cross-Validation (Baseline) ===')
for i, sc in enumerate(cv_scores, 1):
    print(f'  Fold {i}: {sc:.4f}')
print(f'\n  Mean  : {cv_scores.mean():.4f}')
print(f'  Std   : {cv_scores.std():.4f}')

plt.figure(figsize=(8, 4))
plt.bar(range(1, 6), cv_scores, color='#3498db', edgecolor='black')
plt.axhline(cv_scores.mean(), color='red', linestyle='--', label=f'Mean = {cv_scores.mean():.4f}')
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.title('5-Fold CV Accuracy – Baseline XGBoost', fontweight='bold')
plt.ylim(0.85, 1.01)
plt.legend()
for i, v in enumerate(cv_scores):
    plt.text(i+1, v+0.002, f'{v:.3f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('03_cv_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.5 Hyperparameter Tuning (GridSearchCV)

In [ ]:
# Grid parameter untuk tuning
param_grid = {
    'n_estimators'    : [100, 200, 300],
    'max_depth'       : [3, 5, 7],
    'learning_rate'   : [0.05, 0.1, 0.2],
    'subsample'       : [0.7, 0.9],
    'colsample_bytree': [0.7, 0.9]
}

xgb_tuning = XGBClassifier(
    objective='multi:softmax',
    num_class=5,
    random_state=42,
    eval_metric='mlogloss',
    use_label_encoder=False,
    verbosity=0
)

# Untuk efisiensi, gunakan RandomizedSearchCV jika dataset besar
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    estimator=xgb_tuning,
    param_distributions=param_grid,
    n_iter=30,
    scoring='accuracy',
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print('🔍 Memulai Hyperparameter Tuning (Randomized Search)...')
random_search.fit(X_train, y_train_xgb)

print(f'\n✅ Best CV Accuracy : {random_search.best_score_:.4f}')
print(f'Best Parameters    : {random_search.best_params_}')

### 3.6 Training Model Terbaik

In [ ]:
best_params = random_search.best_params_

xgb_best = XGBClassifier(
    objective='multi:softmax',
    num_class=5,
    random_state=42,
    eval_metric='mlogloss',
    use_label_encoder=False,
    verbosity=0,
    **best_params
)

# Training dengan early stopping untuk monitor loss
eval_set = [(X_train, y_train_xgb), (X_test, y_test_xgb)]

xgb_best.fit(
    X_train, y_train_xgb,
    eval_set=eval_set,
    verbose=False
)

y_pred_best = xgb_best.predict(X_test)
acc_best = accuracy_score(y_test_xgb, y_pred_best)

print(f'✅ Best Model Accuracy  (Test Set): {acc_best:.4f} ({acc_best*100:.2f}%)')
print(f'   Baseline Accuracy              : {acc_base:.4f} ({acc_base*100:.2f}%)')
print(f'   Peningkatan                    : +{(acc_best - acc_base)*100:.2f}%')

### 3.7 Learning Curve (Training Loss)

In [ ]:
results = xgb_best.evals_result()
epochs  = len(results['validation_0']['mlogloss'])

plt.figure(figsize=(10, 5))
plt.plot(range(epochs), results['validation_0']['mlogloss'], label='Train Loss', color='#3498db')
plt.plot(range(epochs), results['validation_1']['mlogloss'], label='Test Loss',  color='#e74c3c', linestyle='--')
plt.xlabel('Epoch (n_estimators)')
plt.ylabel('Log Loss')
plt.title('Learning Curve – XGBoost (Multi-Class Log Loss)', fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('03_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Learning curve disimpan.')

### 3.8 Simpan Model

In [ ]:
joblib.dump(xgb_best, 'xgboost_model.pkl')
joblib.dump(best_params, 'best_params.pkl')

print('✅ Model tersimpan:')
print('   - xgboost_model.pkl')
print('   - best_params.pkl')
print(f'\nParameter terbaik:')
for k, v in best_params.items():
    print(f'   {k}: {v}')